In [29]:
import os, json, pickle, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from functions import DoomDataset

In [2]:
CONTEXT = 512
STRIDE  = 256
MAX_TRANSPOSE = 6

BATCH_SIZE = 32
D_MODEL    = 256
N_HEADS    = 8
N_LAYERS   = 4
D_FF       = 1024
DROPOUT    = 0.15     # ważne przy małych danych
LR         = 3e-4
EPOCHS     = 100

PROC_DIR = '../data/processed/'

os.makedirs('../models', exist_ok=True)
os.makedirs('../output', exist_ok=True)

In [3]:
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Device: NVIDIA GeForce RTX 4070


In [4]:
with open(os.path.join(PROC_DIR, 'sequences.pkl'), 'rb') as f:
    data = pickle.load(f)

with open(os.path.join(PROC_DIR, 'vocab.json')) as f:
    token2id = json.load(f)
id2token = {y: x for x,y in token2id.items()}

sequences = data.values()
names = list(data.keys())
VOCAB_SIZE = len(token2id)
PAD_ID = token2id['PAD']
print(f'Sequences: {len(sequences)} | Vocab: {VOCAB_SIZE} | Tokens: {sum(len(s) for s in sequences):,}')

Sequences: 43 | Vocab: 579 | Tokens: 384,327


In [14]:
id2pitch = {}
for tok, i in token2id.items():
    if tok.startswith('NOTE_ON_') or tok.startswith('NOTE_OFF_'):
        id2pitch[i] = int(tok.split('_')[-1])
noteon_ids = { int(tok.split('_')[-1]): i for tok, i in token2id.items() if tok.startswith('NOTE_ON') }
noteoff_ids = { int(tok.split('_')[-1]): i for tok, i in token2id.items() if tok.startswith('NOTE_OFF') }

SHIFT_MAPS = {}
for off in range(-MAX_TRANSPOSE, MAX_TRANSPOSE+1):
    m = np.arange(VOCAB_SIZE)
    for pitch, idx in noteon_ids.items():
        np_ = pitch + off
        if 0 <= np_ <= 127:
            m[idx] = noteon_ids[np_]
    for pitch, idx in noteoff_ids.items():
        np_ = pitch + off
        if 0 <= np_ <= 127:
            m[idx] = noteoff_ids[np_]
    SHIFT_MAPS[off] = m